Task 4: Survey data cleaning (Kaggle survey 2017-2021)

This file has five years of the Kaggle survey stacked on top of each other, around 106 thousand respondents and 293 columns. Every year asked slightly different questions, so most people left most columns empty. In this notebook I clean the data, fix the problems I found, and then use it to answer a few simple questions.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

Loading the file and taking a first look. The first row does not look like a normal respondent.

In [ ]:
df = pd.read_csv('kaggle_survey_2017_2021.csv')
print(df.shape)
df.iloc[:3, :8]

Row 0 is not a person, it holds the full text of each question (for example what Q1 actually asks). I save it in a variable so I can look things up later, and then remove it so only real respondents are left. This should leave 106,301 rows. Only run this cell once, running it twice would remove a real respondent.

In [ ]:
questions = df.iloc[0]
df = df.iloc[1:].copy()
print(df.shape)

Choosing columns. Out of 293 columns, only a few basic questions were asked in every year (age, gender, country, education, job title and years of coding). The rest are specific to one year and mostly empty, so I keep the 7 columns that work across all five years and give them readable names.

In [ ]:
cols = {'-': 'Year', 'Q1': 'Age', 'Q2': 'Gender', 'Q3': 'Country', 'Q4': 'Education', 'Q5': 'JobTitle', 'Q6': 'CodingYears'}
df_clean = df[list(cols)].rename(columns=cols)
print(df_clean.shape)
print(df_clean.isna().sum())

The empty counts for the larger columns (education, job title, coding years) are people who skipped the question, which is normal in a survey. I deal with them in the cleaning step below.

Checking for duplicates. If I only look at these 7 columns I get 42,477 duplicates, but that is misleading because many different people have the same age, country and job. A real duplicate has to match on all the columns, so I check on the full data.

In [ ]:
print(df.duplicated().sum())

dups = df[df.duplicated(keep=False)]
print(dups.notna().sum(axis=1).describe())
print(df.notna().sum(axis=1).describe())

On the full data there are 4,456 rows with an identical earlier copy. But those rows answered a median of only 7 questions, while the average respondent answered 31. So they are mostly people who answered a few basic questions and stopped, and different people can easily have the same few answers by chance. A real double submission would match on dozens of answers. I decided to keep them, because dropping them would remove real respondents.

Now the cleaning. These are the problems I found, all fixed in the next cell:

1. The Year column is a mix of numbers and text (2021 existed both as a number and as the text '2021'), so pandas treated them as two different years. I convert it to numbers.
2. The apostrophes in education are broken (Master's shows up as Masterâ€™s), which also split the same degree into two groups. I repair the text.
3. Some answers mean the same thing but are worded differently in different years: 'I did not complete any formal education past high school' and 'No formal education past high school', 'Professional doctorate' (only in 2021) and 'Professional degree'. I merge each pair. Professional doctorate being the same level as professional degree is my own judgement.
4. The United States is written two ways ('United States' and 'United States of America') and the UK has a very long name. I merge the US and shorten the UK.
5. Blank answers become 'Not answered' so nobody disappears from the counts.
6. Education and coding experience are text, so they cannot be sorted in a real order. I turn them into ordered categories. The survey also used different answer ranges in different years (1-2 years in one, 1-3 in another), so for coding experience I group everything into six buckets that every wording fits into. This is a small approximation. People who skipped, or who said they prefer not to answer, have no place in the order, so they stay empty in the new columns.

In [ ]:
# 1. Year: mix of numbers and text, make it all numbers
df_clean['Year'] = df_clean['Year'].astype(int)

# 2. and 3. repair broken apostrophes and merge answers that mean the same thing
df_clean['Education'] = df_clean['Education'].str.replace('â€™', "'", regex=False)
df_clean['Education'] = df_clean['Education'].replace({
    'I did not complete any formal education past high school': 'No formal education past high school',
    'Professional doctorate': 'Professional degree',
})

# 4. country names
df_clean['Country'] = df_clean['Country'].replace({
    'United States': 'United States of America',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
})

# 5. blanks become an explicit label
cat_cols = ['Age', 'Gender', 'Country', 'Education', 'JobTitle', 'CodingYears']
df_clean[cat_cols] = df_clean[cat_cols].fillna('Not answered')

# 6. Education in real order (answers not in the list become empty)
edu_order = [
    'No formal education past high school',
    "Some college/university study without earning a bachelor's degree",
    "Bachelor's degree",
    "Master's degree",
    'Professional degree',
    'Doctoral degree',
]
edu_valid = df_clean['Education'].where(df_clean['Education'].isin(edu_order))
df_clean['EducationLevel'] = pd.Categorical(edu_valid, categories=edu_order, ordered=True)

# 6. CodingYears: group the different wordings into common buckets
coding_map = {
    'I have never written code': 'Never',
    'I have never written code but I want to learn': 'Never',
    'I have never written code and I do not want to learn': 'Never',
    "I don't write code to analyze data": 'Never',
    '< 1 years': '<1 year', '< 1 year': '<1 year', 'Less than a year': '<1 year',
    '1-2 years': '1-3 years', '1 to 2 years': '1-3 years', '1-3 years': '1-3 years',
    '3-5 years': '3-5 years', '3 to 5 years': '3-5 years',
    '5-10 years': '5-10 years', '6 to 10 years': '5-10 years',
    '10-20 years': '10+ years', 'More than 10 years': '10+ years',
    '20+ years': '10+ years', '20-30 years': '10+ years',
    '30-40 years': '10+ years', '40+ years': '10+ years',
}
code_order = ['Never', '<1 year', '1-3 years', '3-5 years', '5-10 years', '10+ years']
df_clean['CodingLevel'] = pd.Categorical(df_clean['CodingYears'].map(coding_map), categories=code_order, ordered=True)

Checking that the cleaning worked. There should be no empty values in the original 7 columns. EducationLevel is empty for people who did not answer or preferred not to answer, and CodingLevel is empty only for the 13,516 people who did not answer. The years should be five groups adding up to 106,301.

In [ ]:
print(df_clean.isna().sum())
print(df_clean['Year'].value_counts().sort_index())
print(df_clean['Education'].value_counts())
print(df_clean['EducationLevel'].value_counts().sort_index())
print(df_clean['CodingLevel'].value_counts().sort_index())

Now the questions. 

Question 1: has the coding experience of respondents changed over the years? The table shows the percentage of each year's respondents in each experience group (people who did not answer are left out).

In [ ]:
pd.crosstab(df_clean['Year'], df_clean['CodingLevel'], normalize='index').round(3) * 100

There is no clear trend. Only about 4 to 6 percent say they have never written code in every year, and the share of people with under 3 years of coding goes up and down between roughly 41 and 53 percent. So around half of the respondents are fairly new to coding in every year. I would not read much into small differences between years, because the survey changed its answer ranges and a different set of people answers each year.

Question 2: how does coding experience differ by job title? I only look at the 8 most common job titles, because very small groups give unreliable percentages. Each row adds up to 1.

In [ ]:
jobs = df_clean[df_clean['JobTitle'] != 'Not answered']
top = jobs['JobTitle'].value_counts().head(8).index
jobs_top = jobs[jobs['JobTitle'].isin(top)]
pd.crosstab(jobs_top['JobTitle'], jobs_top['CodingLevel'], normalize='index').round(2)

Students are the most junior group, about 73 percent have under 3 years of coding and only 1 percent have more than 10. Research scientists are the most experienced, 49 percent have 5 or more years. Data scientists are clearly more experienced than data analysts: about 65 percent of data scientists have 3 or more years, compared to 42 percent of data analysts, and 53 percent of data analysts have under 3 years. So in this survey data analyst is the more entry level of the two. Business analysts have the highest share of people who have never written code (11 percent) among the named roles, so that job is less code heavy.

Question 3: has the education level of respondents changed over the years?

In [ ]:
pd.crosstab(df_clean['Year'], df_clean['EducationLevel'], normalize='index').round(3) * 100

Master's is the biggest group in every year (between 40 and 47 percent). The share with a bachelor's goes up from 32.2 percent in 2017 to 39.1 percent in 2021, and the share with a doctoral degree goes down from 15.7 to 11.0 percent. This suggests that the people answering the survey are becoming less academic over time. It only describes who answered each year, not the whole field. Also, professional degree dropped to 1.4 percent in 2021, which is probably partly because that year used a different wording for it, so I would be careful comparing that one.

Question 4: where are the respondents from? I skip people who did not answer. 'Other' is not a country, it means the country was too small to be listed separately.

In [ ]:
countries = df_clean[df_clean['Country'] != 'Not answered']['Country'].value_counts()
print(countries.head(15))

India (25,192) and the United States (16,885) are by far the biggest, together about 40 percent of all 106,301 respondents. So anything I say about the survey describes people from these two countries the most. If I had not merged the two ways of writing United States, it would have looked like a much smaller group than it really is.

Summary chart: education level by year, as a share of the people who answered.

In [ ]:
edu = pd.crosstab(df_clean['Year'], df_clean['EducationLevel'], normalize='index') * 100
edu.columns = ['High school or less', 'Some college', "Bachelor's", "Master's", 'Professional', 'Doctoral']

ax = edu.plot(kind='bar', stacked=True, figsize=(9, 5), colormap='viridis')
ax.set_title('Education level of Kaggle survey respondents, 2017-2021')
ax.set_xlabel('Survey year')
ax.set_ylabel('% of respondents who answered')
ax.legend(title='Education', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

The bachelor's part of the bars gets bigger towards 2021 and the doctoral part at the top gets smaller.

What I would keep in mind about this data: it is not one group of people followed over time, it is a different group of volunteers each year, mostly from India and the US, so the results describe who answered the survey and not data people in general.